In [ ]:

import pandas as pd
import numpy as np

INPUT_FILE = "gapminder_after1952.csv"
OUTPUT_FILE = "gapminder_cleaned_for_tableau.csv"

# -----------------------------
# 1) Load data
# -----------------------------
df = pd.read_csv(INPUT_FILE)

# -----------------------------
# 2) Standardize column names
# -----------------------------
def normalize_col(col):
    col = col.strip().lower()
    col = col.replace("(", "").replace(")", "")
    col = col.replace("%", "pct")
    col = col.replace("-", "_").replace(" ", "_")
    while "__" in col:
        col = col.replace("__", "_")
    return col

df.columns = [normalize_col(c) for c in df.columns]

# -----------------------------
# 3) Rename common Gapminder columns
# -----------------------------
rename_map = {
    "country": "country",
    "continent": "continent",
    "year": "year",
    "lifeexp": "life_expectancy",
    "life_expectancy": "life_expectancy",
    "gdppercap": "gdp_per_capita",
    "gdp_per_capita": "gdp_per_capita",
    "pop": "population",
}

df = df.rename(columns={c: rename_map[c] for c in df.columns if c in rename_map})

# -----------------------------
# 4) Clean text columns
# -----------------------------
df["country"] = df["country"].astype(str).str.strip()

if "continent" in df.columns:
    df["continent"] = (
        df["continent"]
        .astype(str)
        .str.strip()
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
    )

# -----------------------------
# 5) Convert numeric columns
# -----------------------------
def to_numeric_safe(series):
    series = series.astype(str).str.replace(",", "", regex=False).str.strip()
    series = series.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return pd.to_numeric(series, errors="coerce")

df["year"] = to_numeric_safe(df["year"]).astype("Int64")

for col in ["life_expectancy", "gdp_per_capita", "population"]:
    if col in df.columns:
        df[col] = to_numeric_safe(df[col])

# -----------------------------
# 6) Filter years (>= 1952)
# -----------------------------
df = df[df["year"].notna()]
df = df[df["year"] >= 1952]

# -----------------------------
# 7) Remove duplicates (country-year)
# -----------------------------
df = df.drop_duplicates(subset=["country", "year"], keep="first")

# -----------------------------
# 8) Sanity checks
# -----------------------------
if "life_expectancy" in df.columns:
    df.loc[(df["life_expectancy"] < 0) | (df["life_expectancy"] > 100), "life_expectancy"] = np.nan

if "gdp_per_capita" in df.columns:
    df.loc[df["gdp_per_capita"] <= 0, "gdp_per_capita"] = np.nan

if "population" in df.columns:
    df.loc[df["population"] <= 0, "population"] = np.nan

# -----------------------------
# 9) Final column selection
# -----------------------------
final_cols = [
    "country",
    "continent",
    "year",
    "life_expectancy",
    "gdp_per_capita",
    "population",
]

final_cols = [c for c in final_cols if c in df.columns]
df = df[final_cols].sort_values(["country", "year"]).reset_index(drop=True)

# -----------------------------
# 10) Save cleaned file
# -----------------------------
df.to_csv(OUTPUT_FILE, index=False)

print("Cleaned file saved as:", OUTPUT_FILE)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()